# Using the shared LLM from a notebook

No model download, no GPU in this notebook: we just call the LLM that the `llm-server` execution hosts for the whole team.

Paste the endpoint from the `llm-server` execution log (`From other workers: http://<private-ip>:11434`).

In [ ]:
LLM_ENDPOINT = "http://10.0.0.0:11434"  # <-- paste the private endpoint here
MODEL = "qwen2.5:1.5b"

In [ ]:
import requests

r = requests.get(f"{LLM_ENDPOINT}/api/tags", timeout=5)
print("Models on the shared server:", [m["name"] for m in r.json()["models"]])

In [ ]:
import time

def ask(prompt, max_tokens=200):
    t0 = time.time()
    r = requests.post(
        f"{LLM_ENDPOINT}/v1/chat/completions",
        json={"model": MODEL, "messages": [{"role": "user", "content": prompt}], "max_tokens": max_tokens},
        timeout=300,
    )
    r.raise_for_status()
    answer = r.json()["choices"][0]["message"]["content"]
    print(f"({time.time() - t0:.1f}s)")
    return answer

print(ask("Explain in two sentences what Valohai does."))

## Same endpoint works with the standard OpenAI client

Ollama's API is OpenAI-compatible, so existing code only needs a different `base_url`.

In [ ]:
%pip install --quiet openai
from openai import OpenAI

client = OpenAI(base_url=f"{LLM_ENDPOINT}/v1", api_key="not-needed")
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Give me 3 bullet points on why to host one LLM centrally."}],
)
print(resp.choices[0].message.content)

## Mini use case: label a few support tickets

In [ ]:
tickets = [
    "My training job has been queued for 2 hours.",
    "Love the new pipeline UI, great work!",
    "How do I add a new S3 bucket as a data store?",
]
for t in tickets:
    label = ask(f"Classify this ticket as BUG, QUESTION or PRAISE. Reply with one word only.\nTicket: {t}", max_tokens=5)
    print(f"{label.strip():10} | {t}")